In [178]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [179]:
np.random.seed(42)

n = 150

temperatura = np.random.uniform(5, 35, n)
personas = np.random.randint(1, 6, n)
horas_casa = np.random.uniform(4, 18, n)

ruido = np.random.normal(0, 1.5, n)

consumo = (
    2
    + 0.08 * (temperatura - 20)**2
    + 1.5 * personas
    + 0.3 * horas_casa
    + ruido
)

df = pd.DataFrame({
    "temperatura": temperatura,
    "personas": personas,
    "horas_casa": horas_casa,
    "consumo": consumo
})

df.head()

,temperatura,personas,horas_casa,consumo
0,16.236204,1,12.790522,11.190949
1,33.521429,1,6.719835,20.749149
2,26.959818,3,4.993173,12.008186
3,22.959755,2,9.554974,8.691399
4,9.680559,5,4.710759,19.044870


In [180]:
X=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [181]:
class SVMRegression:
    def __init__(self,error=1,C=None,kernel='linear',gamma=1, tol=1e-5):
        self.C=C
        self.error=error
        self.kernel=kernel
        self.gamma=gamma
        self.tol=tol

    def Kernel(self,x1,x2):
        if self.kernel=='linear':
            return x1@x2.T
        elif self.kernel=='RBF':
            diff=(x1[:,None]-x2[None,:])**2
            dist=np.sum(diff,axis=2)
            return np.exp(-self.gamma*dist)
        else:
            raise ValueError("Kernel no permitido")
        
    def fit(self,x,y):
        k=self.Kernel(x,x)
        alphas=np.zeros(2*len(y))

        def opt_fun(alphas):
            alphas1=alphas[:len(y)]
            alphas2=alphas[len(y):] 
            diff=alphas1-alphas2
            suma=alphas1+alphas2
            return 0.5*np.sum(diff@k@diff)+self.error*np.sum(suma)-np.sum(diff*y)
        
        cons={"type":'eq',"fun":lambda alphas:np.sum(alphas[:len(y)]-alphas[len(y):])}

        if self.C is None:
            bounds=[(0,None)]*(2*len(y))
            opt=minimize(opt_fun,alphas,bounds=bounds,constraints=cons)
            a=opt.x
            sv=(a>self.tol) 

        else:
            bounds=[(0,self.C)]*(2*len(y))
            opt=minimize(opt_fun,alphas,bounds=bounds, constraints=cons)
            a=opt.x
            sv=(a>self.tol) & (a<self.C-self.tol)
        sv1=sv[:len(y)]
        sv2=sv[len(y):]
        alp=a[:len(y)]
        alp_star=a[len(y):]
        alp_sv=alp[sv1]
        alp_star_sv=alp_star[sv2]

        beta=alp-alp_star 
        c=[] 
        for i in range(len(y)):
            if sv1[i]==True:
                b_i=(y[i]-self.error -np.sum(beta*k[:,i]))
                c.append(b_i)
            elif sv2[i]==True:
                b_i=(y[i]+self.error -np.sum(beta*k[:,i]))
                c.append(b_i)
        b=np.mean(c)
        self.b=b
        self.sv1=sv1
        self.sv2=sv2
        self.beta=beta
        self.x=x

    def predict(self,X):
        k=self.Kernel(X,self.x)
        y_pred=self.b + k@self.beta
        return y_pred


CON KERNEL RBF(PROCESO GAUSSIANO)

In [182]:
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.25, random_state=123)

In [183]:
modelo=SVMRegression(error=1,C=10, kernel='RBF',gamma=0.1)

In [184]:
modelo.fit(x_train,y_train)

In [185]:
y_pred=modelo.predict(x_test)
error_abs=mean_absolute_error(y_test,y_pred)
error_sqr=mean_squared_error(y_test,y_pred)


In [186]:
print(np.std(y))

5.988484468320396


In [187]:
print(f"MAE:{error_abs}")
print(f"MSE:{error_sqr}")

MAE:2.317673038166286
MSE:7.9510498944739885


CON KERNEL LINEAL

In [188]:
modelo1=SVMRegression(error=1,C=10)

In [189]:
modelo1.fit(x_train,y_train)

In [190]:
y_pred1=modelo1.predict(x_test)
error_abs1=mean_absolute_error(y_test,y_pred1)
error_sqr1=mean_squared_error(y_test,y_pred1)

In [191]:
print(f"MAE:{error_abs1}")
print(f"MSE:{error_sqr1}")

MAE:5.094141241530357
MSE:34.03236907220099


In [198]:
xd=pd.DataFrame({"Error absoluto":[error_abs,error_abs1],"Error cuadrado":[error_sqr,error_sqr1]}, index=['RBF', 'Lineal'])
xd.index.name = "Kernel"
xd

,Error absoluto,Error cuadrado
Kernel,,
RBF,2.317673,7.951050
Lineal,5.094141,34.032369
